# 🎯 Multimodal Customer Complaint Classification Using Transformers
### End-to-End Demonstration Notebook

This notebook demonstrates the multimodal deep learning architecture fusing **DistilBERT** text embeddings and **ResNet-18** visual embeddings to predict **Complaint Aspect** and **Complaint Severity**.

In [ ]:
import os
import sys
import json
import torch
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

# Add project root to sys.path
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import config
from src.inference import ComplaintPredictor

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {config.get_device()}")

## 1. Load Processed Dataset & Inspect Distributions

In [ ]:
test_df = pd.read_csv(os.path.join(project_root, 'data', 'processed', 'test.csv'))
print(f"Test set samples: {len(test_df)}")
print("\nAspect Class Counts:")
print(test_df['aspect'].value_counts())
print("\nSeverity Class Counts:")
print(test_df['severity'].value_counts())

## 2. Initialize Multimodal Predictor

In [ ]:
ckpt_path = os.path.join(project_root, 'models', 'multimodal', 'best_model.pt')
predictor = ComplaintPredictor(checkpoint_path=ckpt_path)
print("Predictor successfully initialized!")

## 3. Live Inference on Test Samples

In [ ]:
# Sample 1: Real test sample
sample = test_df.iloc[0]
img_path = os.path.join(project_root, 'data', 'processed', sample['image_path']) if pd.notna(sample['image_path']) else None

print(f"Complaint Text: {sample['text']}")
print(f"Ground Truth -> Aspect: {sample['aspect']} | Severity: {sample['severity']}")

result = predictor.predict(text=sample['text'], image_input=img_path)
print(f"\nPrediction Mode: {result['mode']}")
print(f"Predicted Aspect:   {result['aspect_prediction']} ({result['aspect_confidence']:.2f}%)")
print(f"Predicted Severity: {result['severity_prediction']} ({result['severity_confidence']:.2f}%)")
print(f"Explanation: {result['explanation']}")

if img_path and os.path.exists(img_path):
    img = Image.open(img_path)
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Complaint Screenshot\nGround Truth: {sample['aspect']}")
    plt.axis('off')
    plt.show()

## 4. Test-Only Fallback Demonstration
Demonstrating that when an image is omitted, the model smoothly operates in **Text-only mode** with zero visual feature fallback.

In [ ]:
text_only_complaint = "My screen is totally frozen and won't respond to any touch after updating to the latest iOS release."
res_text = predictor.predict(text=text_only_complaint, image_input=None)
print(f"Mode: {res_text['mode']}")
print(f"Predicted Aspect:   {res_text['aspect_prediction']} ({res_text['aspect_confidence']:.2f}%)")
print(f"Predicted Severity: {res_text['severity_prediction']} ({res_text['severity_confidence']:.2f}%)")

## 5. Evaluation Results and Confusion Matrices

In [ ]:
results_path = os.path.join(project_root, 'outputs', 'results.json')
if os.path.exists(results_path):
    with open(results_path, 'r') as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print("Run python src/evaluate.py to generate full test metrics and confusion matrices.")